# Prototipo — Asistente Inteligente Mercado Central 24h

Notebook de prototipado rápido (recomendado ejecutar en Google Colab).
Úsalo para probar el agente antes de organizarlo en los módulos de `src/`.

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langgraph langchain-text-splitters faiss-cpu pypdf openpyxl

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "PON_AQUI_TU_API_KEY"

## 1. Sube los documentos

En Colab: sube los PDFs de `data/politicas/` y el archivo `inventario_de_supermercado_latam.xlsx` a la sesión, o monta tu Google Drive.

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

loader = PyPDFDirectoryLoader("data/politicas")
documentos = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
fragmentos = splitter.split_documents(documentos)

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = FAISS.from_documents(fragmentos, embeddings)
print(f"Vectorstore listo con {len(fragmentos)} fragmentos")

## 2. Prueba rápida de una consulta al RAG

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
qa = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())

qa.invoke({"query": "¿Cuántos días tengo para devolver un producto?"})

## 3. Siguientes pasos

Una vez validado el prototipo, traslada la lógica a los módulos organizados en `src/` (`ingest.py`, `tools/`, `agent.py`) para el proyecto final.